In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

prompt = [HumanMessage("What is the capital of France?")]

response = model.invoke(prompt)
print(response.content)

The capital of France is Paris.


In [7]:
from langchain_core.messages import SystemMessage, AIMessage 

system_msg  = SystemMessage(
    '''You are a helpful financial advisor that is able to answer complicated financial problems and answer financial question in a simple and understandable manner.'''
)

human_msg = HumanMessage(
    "I want to obtain a credit card. I have been hearing that Wealthsimple and Capital one both have good options for credit cards, as they have high percentage cashback rewards. " \
    "Could you help me with explaining to me which card I should get and why? And if there are any other good options for me."
    )

response = model.invoke([system_msg,human_msg])
print(response.content)

Certainly! When choosing a credit card, it’s important to consider a few key factors: cashback rewards, fees, interest rates, rewards categories, and your spending habits. Let’s take a look at the options you mentioned and a couple of alternatives.

### Wealthsimple Credit Card:
1. **Cashback Rewards**: Wealthsimple typically offers a competitive cashback rate on purchases. For instance, they may offer higher cashback on certain categories like groceries or online purchases.
2. **No Annual Fees**: This card often does not charge an annual fee, which is a great perk if you’re looking to earn rewards without incurring ongoing costs.
3. **Investing Options**: One unique aspect of the Wealthsimple card is that your cashback can be automatically invested into a Wealthsimple account, allowing you to grow your savings passively.

### Capital One Credit Card:
1. **Cashback Rewards**: Capital One tends to have cards that offer cashback in various categories, sometimes with bonus percentages for

In [19]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    """Answer the following questions based off the context I give you. If there is no info on it, just reply with I don't know."
    Context: {context}
    Prompt: {prompt}
    Answer:"""
)

formatted_prompt = template.invoke({
    "context": "The S&P500 historically has around a 15-20 percent annual return rate.",
    "prompt": "What is the historic annual return for the S&P500?"
})

result = model.invoke(formatted_prompt)

print(result.content)

The historic annual return for the S&P500 is around 15-20 percent.


In [21]:
from pydantic import BaseModel

class AnswerWithJustification(BaseModel):
    answer: str
    justificaiton: str 

structured_model = model.with_structured_output(AnswerWithJustification)

response = structured_model.invoke(formatted_prompt)
print(response)

answer='The historic annual return for the S&P500 is around 15-20 percent.' justificaiton='The context provided states that the S&P500 historically has around a 15-20 percent annual return rate.'


In [22]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser() 

items = parser.invoke("Banana, strawberry, apple, fries, soccer, tea, plate, cookie")
print(items)

['Banana', 'strawberry', 'apple', 'fries', 'soccer', 'tea', 'plate', 'cookie']


In [24]:
structured_model.batch(["hey there!", "What day is it?"])

[AnswerWithJustification(answer='Hello! How can I assist you today?', justificaiton='This response is polite and inviting, encouraging further interaction and engagement.'),
 AnswerWithJustification(answer='Today is October 26, 2023.', justificaiton='The date is determined based on the current day of the month and the year.')]

In [37]:
chatbot = template | model

for part in chatbot.stream({
    "context": "The S&P500 historically has around a 15-20 percent annual return rate.",
    "prompt": "What is NASDAQ annual return rate?"
}): 
    print(part.content, end="")

I don't know.